# Lesson 13B: Denoising Diffusion Probabilistic Models - Practical Implementation

## Introduction

**Reference**: Ho, J., Jain, A., & Abbeel, P. (2020). "Denoising Diffusion Probabilistic Models". *Advances in Neural Information Processing Systems (NeurIPS)*, 33, 6840-6851.

Denoising Diffusion Probabilistic Models (DDPMs) are a class of generative models that learn to generate data by reversing a gradual noising process. Unlike GANs or VAEs, diffusion models achieve state-of-the-art sample quality through iterative refinement.

**Key Innovation**: Treating generation as iterative denoising, with theoretical connections to score-based models and stochastic differential equations.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_moons

torch.manual_seed(42)
np.random.seed(42)
print("Libraries loaded successfully")

## Theoretical Foundation

### Forward Diffusion Process

The forward process gradually adds Gaussian noise over T timesteps:

$$q(x_t | x_{t-1}) = \mathcal{N}(x_t; \sqrt{1-\beta_t} x_{t-1}, \beta_t I)$$

where β_t is a variance schedule. We can sample x_t directly from x_0:

$$q(x_t | x_0) = \mathcal{N}(x_t; \sqrt{\bar{\alpha}_t} x_0, (1-\bar{\alpha}_t) I)$$

where $\alpha_t = 1 - \beta_t$ and $\bar{\alpha}_t = \prod_{s=1}^t \alpha_s$.

**Reparameterization**: 
$$x_t = \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1 - \bar{\alpha}_t} \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

### Reverse Diffusion Process

We learn the reverse process p_θ to denoise:

$$p_\theta(x_{t-1} | x_t) = \mathcal{N}(x_{t-1}; \mu_\theta(x_t, t), \Sigma_\theta(x_t, t))$$

For fixed variance Σ_θ = σ_t²I, we only need to learn the mean μ_θ.

### Training Objective

The variational lower bound simplifies to:

$$L = \mathbb{E}_{t, x_0, \epsilon} \left[ \| \epsilon - \epsilon_\theta(x_t, t) \|^2 \right]$$

where ε_θ predicts the noise added at timestep t.

**Key Insight**: Instead of predicting x_0 or μ_θ directly, we predict the noise ε.

### Variance Schedule

The β schedule controls noise addition rate. Common choices:

1. **Linear**: $\beta_t = \beta_1 + \frac{t}{T}(\beta_T - \beta_1)$

2. **Cosine** (Nichol & Dhariwal, 2021): $\bar{\alpha}_t = \frac{f(t)}{f(0)}$, where $f(t) = \cos\left(\frac{t/T + s}{1+s} \cdot \frac{\pi}{2}\right)^2$

We use linear schedule for simplicity.

In [ ]:
def get_beta_schedule(T=100, beta_start=1e-4, beta_end=0.02, schedule_type='linear'):
    """
    Generate variance schedule for diffusion process
    
    Args:
        T: Number of diffusion steps
        beta_start: Initial variance
        beta_end: Final variance
        schedule_type: 'linear' or 'cosine'
    
    Returns:
        betas: Variance schedule [T]
    """
    if schedule_type == 'linear':
        return torch.linspace(beta_start, beta_end, T)
    else:
        raise NotImplementedError(f"Schedule {schedule_type} not implemented")

def get_alpha_schedule(betas):
    """
    Compute α and cumulative product ᾱ
    
    Args:
        betas: Variance schedule [T]
    
    Returns:
        alphas: 1 - β_t [T]
        alphas_cumprod: ᾱ_t = ∏ α_s [T]
    """
    alphas = 1 - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)
    return alphas, alphas_cumprod

# Initialize schedule
T = 100
betas = get_beta_schedule(T, beta_start=1e-4, beta_end=0.02)
alphas, alphas_cumprod = get_alpha_schedule(betas)

print(f"Diffusion configuration:")
print(f"  Timesteps T: {T}")
print(f"  β schedule: linear [{betas[0]:.6f}, {betas[-1]:.6f}]")
print(f"  ᾱ_T: {alphas_cumprod[-1]:.6f} (signal retention at final step)")

# Visualize schedule
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(betas.numpy(), linewidth=2, color='blue')
ax.set_xlabel('Timestep t', fontsize=11)
ax.set_ylabel('β_t', fontsize=11)
ax.set_title('Variance Schedule', fontweight='bold', fontsize=12)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(alphas_cumprod.numpy(), linewidth=2, color='red')
ax.set_xlabel('Timestep t', fontsize=11)
ax.set_ylabel('ᾱ_t', fontsize=11)
ax.set_title('Cumulative Alpha (Signal Retention)', fontweight='bold', fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Forward Diffusion Implementation

We can sample x_t from x_0 in one step using the closed-form expression.

In [ ]:
def forward_diffusion(x0, t, alphas_cumprod):
    """
    Sample x_t from x_0 using reparameterization:
    x_t = √ᾱ_t * x_0 + √(1-ᾱ_t) * ε
    
    Args:
        x0: Initial samples [batch_size, dim]
        t: Timesteps [batch_size]
        alphas_cumprod: ᾱ schedule [T]
    
    Returns:
        x_t: Noised samples [batch_size, dim]
        noise: Added noise [batch_size, dim]
    """
    batch_size = x0.shape[0]
    
    # Get ᾱ_t for each sample
    sqrt_alpha_bar = torch.sqrt(alphas_cumprod[t])
    sqrt_one_minus_alpha_bar = torch.sqrt(1 - alphas_cumprod[t])
    
    # Sample noise
    noise = torch.randn_like(x0)
    
    # Forward diffusion: x_t = √ᾱ_t * x_0 + √(1-ᾱ_t) * ε
    x_t = sqrt_alpha_bar.view(-1, 1) * x0 + sqrt_one_minus_alpha_bar.view(-1, 1) * noise
    
    return x_t, noise

# Demonstrate forward process
X_real, _ = make_moons(n_samples=500, noise=0.05, random_state=42)
X_real = torch.FloatTensor(X_real)

# Visualize noise addition at different timesteps
timesteps = [0, 25, 50, 75, 99]
fig, axes = plt.subplots(1, len(timesteps), figsize=(16, 3))

for i, t_val in enumerate(timesteps):
    t_tensor = torch.full((len(X_real),), t_val, dtype=torch.long)
    X_noisy, _ = forward_diffusion(X_real, t_tensor, alphas_cumprod)
    
    ax = axes[i]
    ax.scatter(X_noisy[:, 0], X_noisy[:, 1], s=10, alpha=0.5)
    ax.set_title(f't = {t_val}\nᾱ_t = {alphas_cumprod[t_val]:.3f}', fontweight='bold')
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.grid(True, alpha=0.3)

plt.suptitle('Forward Diffusion Process q(x_t | x_0)', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()
print("Forward diffusion visualized")

## Denoising Network Architecture

The network ε_θ(x_t, t) predicts the noise ε given noisy input x_t and timestep t.

In [ ]:
class SimpleDiffusion(nn.Module):
    """
    Simplified denoising network for educational purposes
    
    In practice, use U-Net architecture for images
    """
    def __init__(self, data_dim=2, time_dim=32, hidden_dim=128):
        super().__init__()
        
        # Time embedding network
        self.time_embed = nn.Sequential(
            nn.Linear(1, time_dim),
            nn.ReLU(),
            nn.Linear(time_dim, time_dim)
        )
        
        # Denoising network
        self.net = nn.Sequential(
            nn.Linear(data_dim + time_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, data_dim)
        )
    
    def forward(self, x, t):
        """
        Predict noise ε given noisy x_t and timestep t
        
        Args:
            x: Noisy input [batch_size, data_dim]
            t: Timesteps [batch_size]
        
        Returns:
            epsilon: Predicted noise [batch_size, data_dim]
        """
        # Normalize timestep to [0, 1]
        t_normalized = t.unsqueeze(-1).float() / (T - 1)
        
        # Embed timestep
        t_emb = self.time_embed(t_normalized)
        
        # Concatenate and predict
        x_t = torch.cat([x, t_emb], dim=-1)
        return self.net(x_t)

model = SimpleDiffusion(data_dim=2, time_dim=32, hidden_dim=128)
print(f"Denoising network initialized")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## Training Procedure

**Algorithm**: Training Denoising Diffusion Probabilistic Model

```
repeat until convergence:
    x_0 ~ q(x_0)                    # Sample from data
    t ~ Uniform{1, ..., T}          # Sample timestep
    ε ~ N(0, I)                     # Sample noise
    Compute x_t using Eq. (4)       # Forward diffusion
    Compute ε_θ(x_t, t)             # Predict noise
    Update θ via ∇_θ ||ε - ε_θ(x_t, t)||²
```

In [ ]:
# Training configuration
optimizer = optim.Adam(model.parameters(), lr=1e-3)
n_epochs = 500
batch_size = 128

# Prepare dataset
X_train, _ = make_moons(n_samples=2000, noise=0.05, random_state=42)
X_train = torch.FloatTensor(X_train)

# Training loop
losses = []

print(f"Training diffusion model for {n_epochs} epochs...")
model.train()

for epoch in range(n_epochs):
    # Sample batch
    batch_idx = torch.randint(0, len(X_train), (batch_size,))
    batch = X_train[batch_idx]
    
    # Sample random timesteps
    t = torch.randint(0, T, (batch_size,))
    
    # Forward diffusion: compute x_t and true noise
    x_noisy, noise_true = forward_diffusion(batch, t, alphas_cumprod)
    
    # Predict noise
    noise_pred = model(x_noisy, t)
    
    # Compute loss: ||ε - ε_θ(x_t, t)||²
    loss = nn.MSELoss()(noise_pred, noise_true)
    
    # Backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}/{n_epochs}, Loss: {loss.item():.6f}")

print("Training completed")

## Sampling (Reverse Diffusion)

**Algorithm**: Sampling from Denoising Diffusion Probabilistic Model

```
x_T ~ N(0, I)
for t = T, ..., 1 do:
    z ~ N(0, I) if t > 1, else z = 0
    x_{t-1} = 1/√α_t * (x_t - (β_t/√(1-ᾱ_t)) * ε_θ(x_t, t)) + σ_t * z
```

where σ_t² = β_t for simplicity.

In [ ]:
@torch.no_grad()
def sample(model, n_samples, T, betas, alphas, alphas_cumprod):
    """
    Generate samples using reverse diffusion
    
    Args:
        model: Trained denoising network
        n_samples: Number of samples to generate
        T: Number of timesteps
        betas, alphas, alphas_cumprod: Schedules
    
    Returns:
        samples: Generated samples [n_samples, data_dim]
    """
    model.eval()
    
    # Start from pure noise: x_T ~ N(0, I)
    x = torch.randn(n_samples, 2)
    
    # Reverse diffusion
    for t in reversed(range(T)):
        t_tensor = torch.full((n_samples,), t, dtype=torch.long)
        
        # Predict noise
        noise_pred = model(x, t_tensor)
        
        # Compute parameters
        alpha_t = alphas[t]
        alpha_bar_t = alphas_cumprod[t]
        beta_t = betas[t]
        
        # Compute mean of reverse distribution
        mean = (1 / torch.sqrt(alpha_t)) * (
            x - (beta_t / torch.sqrt(1 - alpha_bar_t)) * noise_pred
        )
        
        # Add noise (except at t=0)
        if t > 0:
            noise = torch.randn_like(x)
            sigma_t = torch.sqrt(beta_t)
            x = mean + sigma_t * noise
        else:
            x = mean
    
    return x

# Generate samples
n_samples = 500
samples = sample(model, n_samples, T, betas, alphas, alphas_cumprod)

print(f"Generated {n_samples} samples via reverse diffusion")

## Results Visualization

In [ ]:
# Compare real vs generated distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Real data
ax = axes[0]
ax.scatter(X_train[:, 0], X_train[:, 1], s=20, alpha=0.6, c='blue')
ax.set_xlabel('x₁', fontsize=11)
ax.set_ylabel('x₂', fontsize=11)
ax.set_title('Real Data Distribution', fontweight='bold', fontsize=13)
ax.grid(True, alpha=0.3)

# Generated data
ax = axes[1]
ax.scatter(samples[:, 0], samples[:, 1], s=20, alpha=0.6, c='red')
ax.set_xlabel('x₁', fontsize=11)
ax.set_ylabel('x₂', fontsize=11)
ax.set_title('Generated Distribution (DDPM)', fontweight='bold', fontsize=13)
ax.grid(True, alpha=0.3)

# Training loss
ax = axes[2]
ax.plot(losses, linewidth=1, alpha=0.7)
ax.set_xlabel('Iteration', fontsize=11)
ax.set_ylabel('MSE Loss', fontsize=11)
ax.set_title('Training Loss ||ε - ε_θ||²', fontweight='bold', fontsize=13)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("DDPM successfully learned the data distribution")

## Discussion

### Theoretical Insights

1. **Connection to Score Matching**: The denoising objective is equivalent to score matching:
   $$\nabla_{x_t} \log p(x_t) \approx -\frac{1}{\sqrt{1-\bar{\alpha}_t}} \epsilon_\theta(x_t, t)$$

2. **SDE Formulation** (Song et al., 2021): The diffusion process can be viewed as discretization of:
   $$dx = -\frac{1}{2}\beta(t) x dt + \sqrt{\beta(t)} dw$$

3. **DDIM Sampling** (Song et al., 2020): Deterministic sampling with fewer steps:
   $$x_{t-1} = \sqrt{\bar{\alpha}_{t-1}} \left(\frac{x_t - \sqrt{1-\bar{\alpha}_t} \epsilon_\theta(x_t, t)}{\sqrt{\bar{\alpha}_t}}\right) + \sqrt{1-\bar{\alpha}_{t-1}} \epsilon_\theta(x_t, t)$$

### Advantages of Diffusion Models

1. **Training Stability**: Unlike GANs, no adversarial training
2. **High Sample Quality**: State-of-the-art on image generation (DALL-E 2, Imagen, Stable Diffusion)
3. **Exact Likelihood**: Can compute log p(x) via denoising score matching
4. **Controllability**: Easy to condition on additional inputs

### Limitations

1. **Slow Sampling**: Requires T forward passes (typically 1000 steps)
2. **Computational Cost**: Training requires many denoising steps
3. **Memory**: Storing schedules and intermediate states

### Recent Advances

1. **Classifier-Free Guidance** (Ho & Salimans, 2022): Improved conditional generation
2. **Latent Diffusion** (Rombach et al., 2022): Apply diffusion in latent space (Stable Diffusion)
3. **Cascaded Diffusion** (Ho et al., 2022): Multi-resolution generation
4. **Consistency Models** (Song et al., 2023): One-step generation

## References

1. Ho, J., Jain, A., & Abbeel, P. (2020). "Denoising Diffusion Probabilistic Models". *Advances in Neural Information Processing Systems (NeurIPS)*, 33, 6840-6851.

2. Song, Y., Sohl-Dickstein, J., Kingma, D. P., et al. (2021). "Score-Based Generative Modeling through Stochastic Differential Equations". *International Conference on Learning Representations (ICLR)*.

3. Song, J., Meng, C., & Ermon, S. (2020). "Denoising Diffusion Implicit Models". *arXiv preprint arXiv:2010.02502*.

4. Nichol, A., & Dhariwal, P. (2021). "Improved Denoising Diffusion Probabilistic Models". *International Conference on Machine Learning (ICML)*, 8162-8171.

5. Rombach, R., Blattmann, A., Lorenz, D., et al. (2022). "High-Resolution Image Synthesis with Latent Diffusion Models". *IEEE/CVF Conference on Computer Vision and Pattern Recognition (CVPR)*, 10684-10695.

6. Dhariwal, P., & Nichol, A. (2021). "Diffusion Models Beat GANs on Image Synthesis". *Advances in Neural Information Processing Systems (NeurIPS)*, 34, 8780-8794.